# Notebook 03: Interview Conversation and Follow-up Generation

**Project:** `ofphil-classification-v2`  
**Phase:** 05 — Model Selection and Training (`06_llm_interview_assistant`)  
**Objective:** Implement multi-turn dialogue state tracking, conversation history management, and context-aware follow-up suggestion workflows.

---

## 1. Dialogue State Tracker Implementation

During clinical intake, a Medical Social Worker (MSW) conducts a multi-turn conversation with an applicant. The assistant maintains a structured record of:
- **Turn ID & Timestamp**: Chronological ordering.
- **Speaker Identity**: `MSW` (Social Worker), `PATIENT` (Applicant), or `ASSISTANT_SUGGESTION` (Copilot recommendation).
- **Domain Focus**: Socioeconomic domain being discussed.
- **Identified Ambiguities**: Details flagged for clarification.

In [ ]:
# Dialogue State Data Classes
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Any
import json

@dataclass
class DialogueTurn:
    turn_id: int
    speaker: str  # 'MSW', 'PATIENT', 'ASSISTANT_SUGGESTION'
    text: str
    domain: str
    suggested_followups: List[str] = field(default_factory=list)
    ambiguities_flagged: List[str] = field(default_factory=list)

@dataclass
class InterviewSessionState:
    session_id: str
    patient_anon_id: str
    active_domain: str
    turns: List[DialogueTurn] = field(default_factory=list)
    disclosed_facts: Dict[str, Any] = field(default_factory=dict)
    unresolved_questions: List[str] = field(default_factory=list)
    
    def add_turn(self, speaker: str, text: str, domain: str, followups: List[str] = None, ambiguities: List[str] = None):
        t_id = len(self.turns) + 1
        turn = DialogueTurn(
            turn_id=t_id,
            speaker=speaker,
            text=text,
            domain=domain,
            suggested_followups=followups or [],
            ambiguities_flagged=ambiguities or []
        )
        self.turns.append(turn)
        return turn
    
    def get_conversation_history_text(self) -> str:
        lines = []
        for t in self.turns:
            if t.speaker in ['MSW', 'PATIENT']:
                lines.append(f"{t.speaker}: {t.text}")
        return "\n".join(lines)

print("Dialogue state tracking classes initialized successfully.")

## 2. Ollama Local Client Harness for Follow-up Suggestion

This function packages the conversation context and sends a request to the local Ollama daemon (`http://localhost:11434/api/generate`).

*Note: In this preparation stage, we provide the complete client code. The notebook does not invoke inference until executed manually.*

In [ ]:
# Local Ollama inference client for follow-up suggestions
import requests

def request_ollama_followup(
    session: InterviewSessionState,
    last_patient_utterance: str,
    model_tag: str = "qwen3:8b",
    endpoint: str = "http://localhost:11434",
    timeout: int = 60
) -> Dict[str, Any]:
    """
    Queries the local Ollama instance with conversation history to generate suggested follow-ups.
    Returns structured follow-up recommendations.
    """
    history = session.get_conversation_history_text()
    prompt_text = f"""Conversation History:
{history}

Latest Patient Utterance: "{last_patient_utterance}"

Task: Propose 2 culturally respectful, context-aware follow-up questions in Filipino/Taglish to clarify incomplete or informal income/expenses.
Output valid JSON format with keys: 'detected_ambiguities', 'suggested_followups'.
"""
    payload = {
        "model": model_tag,
        "prompt": prompt_text,
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.2,
            "top_p": 0.9,
            "max_tokens": 512
        }
    }
    
    try:
        resp = requests.post(f"{endpoint}/api/generate", json=payload, timeout=timeout)
        if resp.status_code == 200:
            res_json = resp.json()
            return json.loads(res_json.get('response', '{}'))
        else:
            return {"error": f"HTTP {resp.status_code}: {resp.text}"}
    except requests.exceptions.RequestException as e:
        return {"offline_fallback": True, "message": f"Ollama offline: {e}"}

print("Ollama conversation follow-up client configured.")

## 3. End-to-End Multi-Turn Synthetic Intake Simulation

We stage an end-to-end multi-turn interview workflow using synthetic patient data to demonstrate state tracking and MSW review points without calling any external models.

In [ ]:
# Multi-turn synthetic interview demonstration session
session = InterviewSessionState(
    session_id="SESS-SYNTH-20261124-001",
    patient_anon_id="SYN-PAT-0042",
    active_domain="livelihood_and_expenses"
)

# Turn 1: MSW Opening Question
session.add_turn(
    speaker="MSW",
    text="Magandang araw po. Pag-usapan po natin ang inyong kasalukuyang hanapbuhay at pamilya.",
    domain="livelihood"
)

# Turn 2: Patient Disclosure (Informal livelihood, vague monthly total)
session.add_turn(
    speaker="PATIENT",
    text="Namasada po ako ng traysikel dati ma'am, pero ngayon dahil malabo na ang kanang mata ko, minsan na lang. Mga 200 o 300 pesos lang pag nakakabiyahe, pambili lang ng bigas.",
    domain="livelihood"
)

# Turn 3: Assistant Suggestion (Staged demo showing expected assistant behavior)
session.add_turn(
    speaker="ASSISTANT_SUGGESTION",
    text="[SUGGESTION] Tanungin kung ilang araw sa isang linggo nakakabiyahe, o kung may ibang miyembro ng pamilya na may hanapbuhay.",
    domain="livelihood",
    followups=[
        "Sa isang linggo po, mga ilang araw po kayo karaniwang nakakapamasada?",
        "May iba pa po bang miyembro ng inyong pamilya na tumutulong sa gastusin?"
    ],
    ambiguities=["Unclear weekly frequency of tricycle driving", "Unstated alternative household income sources"]
)

print(f"Synthetic interview state tracked. Total turns logged: {len(session.turns)}")
print("\n--- Current Transcript ---")
print(session.get_conversation_history_text())